## A. Processus de collecte et d'extraction  
- *source* -> questionnaire type (simplifié) d'enquête multisectorielle semi-administré aux ménages : deux onglets dont un nommé "Choix" contenant le codage des réponses en choix unique ou multiples. 
- *outil de collecte* -> kobotoolbox  https://eu.kobotoolbox.org/
- *extraction* -> fichier en format XLS - puis conversion en CSV (prompt) 
- *qualité à l'extraction* -> fichier brut pouvant contenir des erreurs (biais)
- *facilité de traitement* -> nécessité d'extraire l'onglet choix du questionnaire pour pouvoir déchiffrer les réponses

## B. Audit initial

In [42]:
import pandas as pd
import numpy as np
df_brute=pd.read_csv(r"C:\Users\maril\Documents\Enquête Multisectorielle - ONG\data\enquete_vul_brute.csv")
df_brute.head(10)

,date_enquete,heure_debut,heure_fin,enqueteur,village,code_jeton,presence_maison,volont,group_info_geo,enquete,...,type_eau,terre,outims,asso,protection,depl_bnf,depl_dist,diff_deplac,type_diff,remarques_enquete
0,2024-04-12,13:30,14:11,enqueteur_7,village_5,99343,oui,non,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-03-11,12:01,12:17,enqueteur_23,village_7,43227,oui,oui,B,B,...,eau_5,non,non,non,A,non,4853.21,OUI,autre coutrans contrasoc,le ménage est très vulnérable suite aux inonda...
2,2024-09-27,10:17,10:53,enqueteur_16,village_11,77589,oui,oui,B,A,...,eau_1,non,non,oui,A,oui,1595.81,non,pproblsec contrasoc,nous avons besoin de manioc et de semences
3,2024-04-16,14:53,16:13,enqueteur_2,village_8,14909,oui,oui,A,A,...,eau_2,non,oui,oui,A,oui,3089.75,non,NaN,nous avons encore été attaqués la nuit dernière
4,2024-03-12,14:20,15:22,enqueteur_20,autre,25302,oui,oui,B,B,...,eau_3,non,non,ERREUR,B,non,3098.19,oui,NaN,difficultés d'accès aux points d'eau potable a...
5,2025-12-01,14:24,15:31,enqueteur_2,village_6,68850,oui,oui,A,B,...,eau_6,non,oui,non,A,non,-99.00,oui,OUI,perte totale des stocks de vivres lors des der...
6,2024-01-21,13:12,13:27,enqueteur_22,village_12,79328,oui,oui,A,B,...,eau_4,non,non,non,A,non,2149.50,non,incphy coutrans autre,difficultés d'accès aux points d'eau potable a...
7,2025-09-06,11:35,12:43,enqueteur_10,autre,71308,oui,oui,A,A,...,eau_2,oui,non,non,B,oui,4138.78,oui,coutrans,pas d'assistance reçue depuis le début de la c...
8,2024-05-01,16:16,17:09,enqueteur_23,village_13,17284,oui,oui,B,A,...,eau_5,oui,non,oui,A,oui,3622.90,non,incphy,perte totale des stocks de vivres lors des der...
9,2025-04-11,16:45,17:09,enqueteur_18,village_7,91988,oui,oui,A,B,...,eau_4,non,non,oui,A,non,4492.11,oui,coutrans,nous attendons une aide en relance agro


In [45]:
print("\n=== RAPPORT D'AUDIT")

print("\n=== types des colonnes ===")
pd.DataFrame({
    "Type": df_brute.dtypes.value_counts().index.astype(str),
    "Nombre_de_colonnes": df_brute.dtypes.value_counts().values,
    "Liste_des_colonnes": [
        ", ".join(df_brute.select_dtypes(include=[t]).columns) 
        for t in df_brute.dtypes.value_counts().index
    ]
})


=== RAPPORT D'AUDIT

=== types des colonnes ===


,Type,Nombre_de_colonnes,Liste_des_colonnes
0,str,73,"date_enquete, heure_debut, heure_fin, enqueteu..."
1,float64,44,"age, loyer_depl_montant, loyer_auto_montant, h..."
2,int64,1,code_jeton


In [47]:
print("\n=== noms des colonnes ===")
cols = df_brute.columns.tolist()
for i in range(0, len(cols), 10):
  print(", ".join(cols[i : i + 10]))

print("\n=== synthèse ===")
print(f"Données collectées | {df_brute.shape[0]:6} | {df_brute.shape[1]} colonnes")
print(f"Données manquantes | {df_brute.isnull().sum().sum()}")
print(f"Données négatives | {df_brute.select_dtypes(include='number').lt(0).sum().sum()}")

print(f"Doublons : enquêtes identiques | {df_brute.duplicated().sum()}")
print(f"Doublons : jetons identiques | {df_brute['code_jeton'].duplicated().sum()}")


=== noms des colonnes ===
date_enquete, heure_debut, heure_fin, enqueteur, village, code_jeton, presence_maison, volont, group_info_geo, enquete
acpt_partage, repondant, groupe_info, age, sexe, poss_carte, typedepiece, statmtr, decision, group_statut_menage
status, herber, environ, date_deplacement, date_retour, loyer_depl, loyer_depl_montant, loyer_auto, loyer_auto_montant, h
h_0_5, h_6_24, h_25_59, h_5_14, h_15_17, h_18_49, h_50_59, h_plus60, f, f_0_5
f_6_24, f_25_59, f_5_14, f_15_17, f_18_49, f_50_59, f_plus60, fem_enc, fem_all, washington
questionnaire_washington, vis_ss, hear_ss, mob_ss, cog_ss, sc_ss, com_ss, observation_0, adultincap, hand
malnut, malnut_charge, malnut_charge_non, source_revenu, source_reve, revenu, dette, montant_dette, repas_a, repas_e
ressources, repas_r, suffisant, repas_f, manque, repas_m, AME, nb_bidon, nb_cass, nb_bas
nb_outil, nb_couch, nb_couv, nb_habit_femme, nb_habit_e, nb_cale, nb_jarre, nb_seau, nb_pot, recipient
type_puisage, capacite, type_stocka

In [73]:
print("\n== répartition des valeurs manquantes au sein des colonnes ==")
missing_top = df_brute.isnull().sum().sort_values(ascending=False)
print(missing_top[missing_top > 0].head(10))


== répartition des valeurs manquantes au sein des colonnes ==
loyer_depl_montant    26349
loyer_auto_montant    26176
f_25_59                4099
h_plus60               4097
revenu                 4089
nb_habit_e             4087
repas_r                4082
nb_couv                4079
h_0_5                  4076
repas_m                4076
dtype: int64


In [40]:
print("\n== valeurs min et max des variables numériques ==")
df_brute.select_dtypes(include='number').agg(['min', 'max']).T.head(15)


== valeurs min et max des variables numériques ==


,min,max
code_jeton,10000.0,99998.0
age,-99.0,999999.0
loyer_depl_montant,-99.0,999999.0
loyer_auto_montant,-99.0,999999.0
h_0_5,-99.0,999999.0
h_6_24,-99.0,999999.0
h_25_59,-99.0,999999.0
h_5_14,-99.0,999999.0
h_15_17,-99.0,999999.0
h_18_49,-99.0,999999.0


In [56]:
print("\n== Eléments de réponses différents au sein des variables catégoriques ==")
print(df_brute.select_dtypes(include=['object', 'string']).nunique().sort_values(ascending=False))


== Eléments de réponses différents au sein des variables catégoriques ==
date_deplacement    1003
date_agro           1003
date_marai          1003
date_retour         1003
date_vivri          1003
                    ... 
outims                 5
diff_deplac            5
depl_bnf               5
presence_maison        2
volont                 2
Length: 73, dtype: int64


## C. Audit métier

### Valeurs manquantes

In [75]:
print("\n== Croisement des valeurs manquantes vs consentement ==")
seuil_vides = df_brute.isnull().mean(axis=1) > 0.9
lignes_vides = df_brute[seuil_vides]

print(f"Nombre de lignes vides à plus de 90% : {len(lignes_vides)}")

# Vérification croisée avec la variable de consentement 
if "volont" in df_brute.columns:
  nb_refus = lignes_vides["volont"].value_counts().get("non", 0)
  print(
      "Nombre de répondants ne souhaitant pas prendre part à l'enquête"
      f" (Entretient clot) : {nb_refus}"
  )

print("\n== Croisement des valeurs manquantes vs réponses sur le loyer (question conditionnée) - loyer_depl_montant ==")
nb_nan = df_brute['loyer_depl_montant'].isnull().sum()
print(f"Nombre de valeurs manquantes dans la colonne : {nb_nan}")
if "loyer_depl" in df_brute.columns and "loyer_depl_montant" in df_brute.columns:
  justifies_loyer = df_brute[
      (df_brute["loyer_depl"] == "non")
      & (df_brute["loyer_depl_montant"].isnull())
  ]
  print(
      "Absence de montant justifié (le répondant a déclaré précédemment ne pas"
      f" payer de loyer) : {len(justifies_loyer)}"
  )

print("\n== Croisement des valeurs manquantes vs réponses sur le loyer (question conditionnée) - loyer_auto_montant ==")
nb_nan = df_brute['loyer_auto_montant'].isnull().sum()
print(f"Nombre de valeurs manquantes dans la colonne : {nb_nan}")
if "loyer_auto" in df_brute.columns and "loyer_auto_montant" in df_brute.columns:
  justifies_loyer = df_brute[
      (df_brute["loyer_auto"] == "non")
      & (df_brute["loyer_auto_montant"].isnull())
  ]
  print(
      "Absence de montant justifié (le répondant a déclaré précédemment ne pas"
      f" payer de loyer) : {len(justifies_loyer)}"
  )



== Croisement des valeurs manquantes vs consentement ==
Nombre de lignes vides à plus de 90% : 3365
Nombre de répondants ne souhaitant pas prendre part à l'enquête (Entretient clot) : 3365

== Croisement des valeurs manquantes vs réponses sur le loyer (question conditionnée) - loyer_depl_montant ==
Nombre de valeurs manquantes dans la colonne : 26349
Absence de montant justifié (le répondant a déclaré précédemment ne pas payer de loyer) : 21517

== Croisement des valeurs manquantes vs réponses sur le loyer (question conditionnée) - loyer_auto_montant ==
Nombre de valeurs manquantes dans la colonne : 26176
Absence de montant justifié (le répondant a déclaré précédemment ne pas payer de loyer) : 21272


En croisant la première observation des valeurs manquantes avec le sens métier du questionnaire, on observe que pour des lignes comprenant plus de 90% d'ommissions de réponses, 100% d'entre elles sont en réalité des entretiens où le répondant n'a pas souhaité prendre part à l'enquête. En masquant ces lignes au nettoyage, on pourra se rendre compte de la vrai proportion des valeurs manquantes et décider de la stratégie de complétion. 

Pour les colonnes loyer_depl_montant et loyer_auto_montant qui observait un nombre considérable par rapport à la moyenne, on peut croiser avec le sens métier de la colonne qui dépent de la question précédente : "payez-vous un loyer ?", si la réponse est non alors automatiquement dans le questionnaire, par conditionnement, la question suivante ne sera pas posée et donc la réponse au moment de l'extraction brute sera vide. Au moment du nettoyage, ces cases vides (du au conditionnement) devront être laissées quand même. Cependant le résidu de valeurs manquantes (personnes déclarant payer des loyers) doit être analysées et traité différement (ênquête interrompue, réelle donnée manquante etc.)

Cela montre qu'au moment du nettoyage chaque conditionnement du questionnaire donc être pris en considération afin de réaliser un nettoyage milimétré.

_________________________________________

Pour ce rendre compte des biais du quesitonnaire (dans son conditionnement qui pousse à des erreurs de réponse) on peut également regarder si, des personnes déclarants ne pas payer de loyer, on quand même eu la possibilité de renseigner un montant : 

In [76]:
if 'loyer_depl' in df_brute.columns and 'loyer_depl_montant' in df_brute.columns:
    anomalies_loyer = df_brute[
        (df_brute['loyer_depl'] == 'non') & 
        (df_brute['loyer_depl_montant'].notnull())
    ]
    print(f"Incohérences loyer (montant renseigné alors que pas de loyer déclaré) : {len(anomalies_loyer)}")

Incohérences loyer (montant renseigné alors que pas de loyer déclaré) : 752


### Valeurs Extrêmes

Dans les colonnes contenant des valeurs extrêmes tel que -99 ou 99999 on peut apporter un contexte métier : 
- -99 est souvent renseigné par les enquêteurs quand la réponse n'est pas connue du répondant 
- 999999 est souvent une erreur informatique

Ici, pour les colonnes : code_jeton, age, et la composition par genre de ménages, il s'agit d'une erreur de contrainte du questionnaire (le jeton est habituellement un code standard contraint à 4 caractères, l'âge est contraint entre 0 et 100, et la composition des ménages par genre et par tranches d'âge se limite à 20 personnes par sous-groupe). 

Pour les loyers il est plus interressant de regarder les outliers et pour l'âge regardé si des mineurs ont répondu (ce qui dans la plus part des cas n'est pas autorisé) :

In [80]:
if 'loyer_depl_montant' in df_brute.columns:
    loyers_valides = df_brute[(df_brute['loyer_depl_montant'] > 0) & (df_brute['loyer_depl_montant'] != 999999)]
    
    q1 = loyers_valides['loyer_depl_montant'].quantile(0.25)
    q3 = loyers_valides['loyer_depl_montant'].quantile(0.75)
    iqr = q3 - q1
    borne_sup = q3 + (1.5 * iqr)
    
    outliers_loyer = loyers_valides[loyers_valides['loyer_depl_montant'] > borne_sup]
    print(f"Outliers détectés sur le loyer déplacement : {len(outliers_loyer)} (Seuil statistique > {borne_sup:.2f})")

if 'loyer_auto_montant' in df_brute.columns:
    loyers_valides = df_brute[(df_brute['loyer_auto_montant'] > 0) & (df_brute['loyer_auto_montant'] != 999999)]
    
    q1 = loyers_valides['loyer_auto_montant'].quantile(0.25)
    q3 = loyers_valides['loyer_auto_montant'].quantile(0.75)
    iqr = q3 - q1
    borne_sup = q3 + (1.5 * iqr)
    
    outliers_loyer = loyers_valides[loyers_valides['loyer_auto_montant'] > borne_sup]
    print(f"Outliers détectés sur le loyer déplacement : {len(outliers_loyer)} (Seuil statistique > {borne_sup:.2f})")

if 'age' in df_brute.columns:
    # Exclusion des codes d'erreur (-99) pour cibler les vrais âges saisis
    mineurs = df_brute[(df_brute['age'] >= 0) & (df_brute['age'] < 18)]
    print(f"Nombre de répondants mineurs (âge < 18 ans) : {len(mineurs)}")
    
    if len(mineurs) > 0:
        print(f"Âge minimum relevé chez les mineurs : {mineurs['age'].min()}")

Outliers détectés sur le loyer déplacement : 0 (Seuil statistique > 146664.31)
Outliers détectés sur le loyer déplacement : 0 (Seuil statistique > 147887.24)
Nombre de répondants mineurs (âge < 18 ans) : 1676
Âge minimum relevé chez les mineurs : 15.0


### Variabilités des réponses

Pour ce qui est de la répétition des réponses il peut s'agir en réalité : d'une étendue (période), d'une représentativité du nombre de choix possibles via le questionnaire, ou d'une réponse en text ouvert (voulue et à analysée ou non-voulue dûes à des erreurs de contraintes dans le questionnaire et donnant lieux à des multiples réponses). 

Les colonnes dates : 

In [84]:
cols_dates = ['date_deplacement', 'date_agro', 'date_marai', 'date_retour', 'date_vivri']

for col in cols_dates:
    if col in df_brute.columns:
        # Conversion temporaire en datetime 
        min_date = pd.to_datetime(df_brute[col], errors='coerce').min()
        max_date = pd.to_datetime(df_brute[col], errors='coerce').max()
        print(f"{col} -> Min : {min_date.date()} | Max : {max_date.date()}")

date_deplacement -> Min : 2020-01-01 | Max : 2022-09-26
date_agro -> Min : 2020-01-01 | Max : 2022-09-26
date_marai -> Min : 2020-01-01 | Max : 2022-09-26
date_retour -> Min : 2020-01-01 | Max : 2022-09-26
date_vivri -> Min : 2020-01-01 | Max : 2022-09-26


Ici l'étendue marque un problème de cohérence dans les dates puisqu'il parait presque impossible qu'une famille déplacée retourne le même jour chez elle. Il faudra regarder également dans la cohérence si des dates de retour ne sont pas antérieures aux dates de déplacement ou encore si certaine famille autochtone ont déclaré un déplacement alors que la question était conditionnée pour n'être répondue uniquement pas les familles déplacées ou réfugiées. 

In [85]:
if "date_deplacement" in df_brute.columns and "date_retour" in df_brute.columns:
    # Conversion temporaire à la volée pour comparer l'ordre chronologique
    dep = pd.to_datetime(df_brute["date_deplacement"], errors="coerce")
    ret = pd.to_datetime(df_brute["date_retour"], errors="coerce")
    
    incoherences_dates = df_brute[ret < dep]
    print(f"Nombre d'incohérences (date de retour antérieure au déplacement) : {len(incoherences_dates)}")

Nombre d'incohérences (date de retour antérieure au déplacement) : 21177


NB : Au-delà des problèmes techniques de conditionnement ou de contraintes du questionnaire d'origine, la traduction, la compréhension de l'enquêteur, la compréhension de l'enquêté et l'interprétation et la vérification logique de la réponse par l'enquêteur entrent également en compte dans la qualité de la donnée récolté. 

___________________________________________________________

Vérifier quelles colonnes ont un nombre de réponses différentes en comparaison aux choix disponibles : 

In [88]:
df_choix = pd.read_csv(r"C:\Users\maril\Documents\Enquête Multisectorielle - ONG\data\code_choix.csv")

# Calcul du nombre de choix théoriques attendus par variable
choix_theoriques = df_choix.groupby("list_name")["code"].nunique()

# Comparaison avec les données observées dans df_brute
for col, nb_theo in choix_theoriques.items():
  if col in df_brute.columns:
    nb_obs = df_brute[col].dropna().nunique()
    if nb_obs != nb_theo:
      print(
          f"Incohérence sur '{col}' : {nb_obs} modalités observées vs"
          f" {nb_theo} attendues dans le codebook."
      )

Incohérence sur 'culture' : 6 modalités observées vs 3 attendues dans le codebook.
Incohérence sur 'environ' : 7 modalités observées vs 4 attendues dans le codebook.
Incohérence sur 'raison' : 43 modalités observées vs 4 attendues dans le codebook.
Incohérence sur 'recipient' : 5 modalités observées vs 9 attendues dans le codebook.
Incohérence sur 'sexe' : 5 modalités observées vs 2 attendues dans le codebook.


Pour la colonne raison , le questionnaire propose une réponse multiple dont le nombre exponentiel de modalités observées par rapport à celles attendues. 
Pour la colonne récipient il s'agit également d'une réponse multiple mais certaines ne semblent pas avoir été selectionnées ce qui pourra peut être montrer à l'analyse que certains recipient sont manquants dans les foyers. 

Par contre, pour les autres, il s'agit de choix unique, le nombre de réponses devraient correspondre, cela s'explique donc pas un manque de contraintes du questionnaire : 

In [92]:
cols_a_verifier = ['sexe', 'environ','culture']

for col in cols_a_verifier:
    if col in df_brute.columns:
        modalites = df_brute[col].dropna().unique()
        print(f"Modalités pour '{col}' ({len(modalites)} unées) : {list(modalites)}")

Modalités pour 'sexe' (5 unées) : ['femme', 'homme', 'OUI', 'non ', 'ERREUR']
Modalités pour 'environ' (7 unées) : ['non ', 'environ_2', 'environ_3', 'environ_4', 'environ_1', 'ERREUR', 'OUI']
Modalités pour 'culture' (6 unées) : ['cult3', 'cult2', 'cult1', 'ERREUR', 'non ', 'OUI']


Dans la même optique d'erreur on va chercher les colonnes normalement de type bool, qui correspondent à un choix de réponse en Oui ou Non mais qui contiennent des erreurs. L'identification de ces colonnes se fait en masquant les colonnes ayant des correspondances avec le fichier choix, les colonnes float, int et date (str), ainsi que des colonnes qui correspondent à la sefmentation des différentes sections du questionnaire. 

In [95]:
colonnes_binaires_cibles = [
    'presence_maison', 'volont', 'acpt_partage', 'repondant', 'poss_carte', 
    'loyer_depl', 'loyer_auto', 'fem_enc', 'fem_all', 'observation_0', 
    'hand', 'malnut', 'malnut_charge', 'source_reve', 'dette', 
    'ressources', 'suffisant', 'manque', 'separer', 'agro', 
    'marai', 'vivri', 'membre', 'eau_point', 'terre', 
    'outims', 'asso', 'depl_bnf', 'diff_deplac'
]

# Audit des anomalies de contraintes sur ces 29 variables
print("=== Audit des variables catégorielles cibles ===")
for col in colonnes_binaires_cibles:
    if col in df_brute.columns:
        # Extraction des valeurs uniques en ignorant les NaN et les codes sentinelles (-99)
        vals_uniques = df_brute[col].dropna().unique()
        vals_propres = [v for v in vals_uniques if str(v) not in ['-99', '-99.0', '999999', '999999.0', 'nan']]
        
        nb_modalites = len(vals_propres)
        
        # Alerte si le nombre de modalités dépasse un seuil suspect (ex: > 2 ou > attendu)
        statut = "OK" if nb_modalites <= 2 else "ANOMALIE"
        print(f"[{statut}] {col} : {nb_modalites} modalités trouvées -> {vals_propres}")
    else:
        print(f"[ERREUR] La colonne '{col}' est absente du dataset.")

=== Audit des variables catégorielles cibles ===
[OK] presence_maison : 2 modalités trouvées -> ['oui', 'non']
[OK] volont : 2 modalités trouvées -> ['non', 'oui']
[ANOMALIE] acpt_partage : 5 modalités trouvées -> ['non', 'oui', 'OUI', 'non ', 'ERREUR']
[ANOMALIE] repondant : 5 modalités trouvées -> ['oui', 'non', 'ERREUR', 'non ', 'OUI']
[ANOMALIE] poss_carte : 5 modalités trouvées -> ['oui', 'non', 'OUI', 'ERREUR', 'non ']
[ANOMALIE] loyer_depl : 5 modalités trouvées -> ['non', 'non ', 'OUI', 'oui', 'ERREUR']
[ANOMALIE] loyer_auto : 5 modalités trouvées -> ['oui', 'non', 'non ', 'OUI', 'ERREUR']
[ANOMALIE] fem_enc : 5 modalités trouvées -> ['non', 'oui', 'non ', 'ERREUR', 'OUI']
[ANOMALIE] fem_all : 5 modalités trouvées -> ['oui', 'non', 'ERREUR', 'OUI', 'non ']
[ANOMALIE] observation_0 : 6 modalités trouvées -> ['pas', 'oui', 'non', 'non ', 'ERREUR', 'OUI']
[ANOMALIE] hand : 5 modalités trouvées -> ['oui', 'non', 'ERREUR', 'OUI', 'non ']
[ANOMALIE] malnut : 5 modalités trouvées -> [